In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

dl_lab_2_stuff_detection_path = kagglehub.competition_download('dl-lab-2-stuff-detection')
packagemanager_pm_110859939_at_03_01_2026_13_55_34_path = kagglehub.notebook_output_download('packagemanager/pm-110859939-at-03-01-2026-13-55-34')

print('Data source import complete.')


In [ ]:
from ultralytics import YOLO

# Load a model
model = YOLO("yolo26n.pt")

## Процесс обучения очень простой

In [ ]:
results = model.train(data=r"/kaggle/input/dl-lab-2-stuff-detection/data.yaml", epochs=5, imgsz=640, name='lab2', project='miet')

# Загружаем нашу лучшую модель

In [ ]:
model = YOLO('/kaggle/working/runs/detect/miet/lab2/weights/best.pt')

In [ ]:
results = model.predict(r'/kaggle/input/dl-lab-2-stuff-detection/test_images/test_images', save_txt=True, save_conf=True, stream=True)
for r in results:
    pass

# Собираем итоговое предсказание

In [ ]:
from pathlib import Path
import json
import pandas as pd


def build_submission_from_solution_order(
    solution_csv: str,
    preds_dir: str,
    output_csv: str = "submission.csv",
    image_col: str = "image_name",
    boxes_col: str = "boxes",
    row_id_col: str = "id",
    require_score: bool = True,
    clamp_score: bool = True,
    keep_only_class: int | None = None,  # например 1, если нужны только строки класса 1; None = все классы
) -> None:
    """
    Builds submission.csv in EXACTLY the same image_name order as solution.csv.

    - Reads solution_csv, takes image_name column order as ground truth order.
    - For each image_name, looks for preds_dir/<stem>.txt
      where stem is Path(image_name).stem
    - If missing -> boxes=[]
    - Prediction txt lines: class xc yc w h score
    - Output boxes: JSON [[xc,yc,w,h,score], ...]
    """
    sol_path = Path(solution_csv)
    pdir = Path(preds_dir)

    if not sol_path.exists():
        raise FileNotFoundError(f"solution_csv not found: {sol_path}")
    if not pdir.exists() or not pdir.is_dir():
        raise FileNotFoundError(f"preds_dir not found or not a dir: {pdir}")

    sol = pd.read_csv(sol_path)
    if image_col not in sol.columns:
        raise ValueError(f"solution.csv must contain column '{image_col}'")

    # Keep original order from solution
    image_names = sol[image_col].astype(str).tolist()

    rows = []
    for idx, image_name in enumerate(image_names):
        stem = Path(image_name).stem
        pred_file = pdir / f"{stem}.txt"

        boxes = []
        if pred_file.exists():
            content = pred_file.read_text(encoding="utf-8", errors="replace").strip()
            if content:
                for ln in content.splitlines():
                    ln = ln.strip()
                    if not ln:
                        continue
                    parts = ln.split()
                    if require_score and len(parts) < 6:
                        continue
                    if len(parts) < 5:
                        continue

                    try:
                        cls = int(float(parts[0]))
                        if keep_only_class is not None and cls != keep_only_class:
                            continue

                        xc = float(parts[1])
                        yc = float(parts[2])
                        w = float(parts[3])
                        h = float(parts[4])
                        sc = float(parts[5]) if len(parts) >= 6 else 1.0
                    except ValueError:
                        continue

                    if clamp_score:
                        sc = 0.0 if sc < 0.0 else (1.0 if sc > 1.0 else sc)

                    boxes.append([xc, yc, w, h, sc])

        rows.append(
            {
                row_id_col: idx,
                image_col: image_name,  # EXACT match
                boxes_col: json.dumps(boxes, separators=(",", ":")),
            }
        )

    sub = pd.DataFrame(rows, columns=[row_id_col, image_col, boxes_col])
    sub.to_csv(output_csv, index=False)
    print(f"Saved: {output_csv} ({len(sub)} rows). Missing preds filled with [] in solution order.")


# пример запуска:


In [ ]:
build_submission_from_solution_order(
    solution_csv=r"/kaggle/input/dl-lab-2-stuff-detection/sample_sub.csv",
    preds_dir=r"/kaggle/working/runs/detect/predict/labels",
    output_csv="submission.csv",
    keep_only_class=1,  # если нужно брать только класс 1; если нет — поставь None
)